In [11]:
import json
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent
DATA_PATH = BASE_DIR / "data" / "corpus.json"

with open(
    DATA_PATH,
    "r",
    encoding="utf-8"
) as file:
    corpus = json.load(file)

print("Dataset loaded successfully.")
print("Root type:", type(corpus))

Dataset loaded successfully.
Root type: <class 'dict'>


In [12]:
print("Root keys:")

for key in corpus.keys():
    print("-", key)

Root keys:
- graine
- reunions
- compte


In [13]:
for key, value in corpus.items():

    print("\n" + "=" * 60)
    print("KEY:", key)
    print("TYPE:", type(value))

    if isinstance(value, list):

        print("LENGTH:", len(value))

        if len(value) > 0:
            print("FIRST ITEM:")
            print(value[0])

    elif isinstance(value, dict):

        print("NUMBER OF KEYS:", len(value))
        print("KEYS:")

        for subkey in value.keys():
            print("  -", subkey)

    else:

        print("VALUE:", value)


KEY: graine
TYPE: <class 'int'>
VALUE: 20260818

KEY: reunions
TYPE: <class 'list'>
LENGTH: 8
FIRST ITEM:
{'id': 'M1', 'societe': 'Northgate Hardware', 'type': 'weekly operations meeting', 'date': '2026-08-03', 'participants': ['Rafael', 'Nadia', 'Colin', 'Padma'], 'repliques': [{'qui': 'Rafael', 'texte': 'Right, let us get started, I have a hard stop at eleven.'}, {'qui': 'Nadia', 'texte': 'Two people are off next week, so we will be short handed.'}, {'qui': 'Rafael', 'texte': 'I will update the pricing on the kiosk by Wednesday.'}, {'qui': 'Colin', 'texte': 'I will take down the summer banner.'}, {'qui': 'Colin', 'texte': 'Nadia, can you call the sign shop about the awning?'}, {'qui': 'Nadia', 'texte': 'Sure, that is mine.'}, {'qui': 'Nadia', 'texte': 'Agreed, let us keep it short today.'}, {'qui': 'Padma', 'texte': 'And the mailbox at the old address still has to be emptied, that one is still open.'}, {'qui': 'Colin', 'texte': 'Should we put a neon sign in the window?'}, {'qui': 'P

In [14]:
def inspect_structure(
    obj,
    level=0,
    max_level=4
):

    indent = "  " * level

    if level > max_level:
        return

    if isinstance(obj, dict):

        print(
            f"{indent}DICT ({len(obj)} keys)"
        )

        for key, value in obj.items():

            print(
                f"{indent}{key}: {type(value).__name__}"
            )

            if level < max_level:

                inspect_structure(
                    value,
                    level + 1,
                    max_level
                )

    elif isinstance(obj, list):

        print(
            f"{indent}LIST ({len(obj)} items)"
        )

        if obj:

            inspect_structure(
                obj[0],
                level + 1,
                max_level
            )

    else:

        print(
            f"{indent}VALUE: {str(obj)[:150]}"
        )


inspect_structure(corpus)

DICT (3 keys)
graine: int
  VALUE: 20260818
reunions: list
  LIST (8 items)
    DICT (7 keys)
    id: str
      VALUE: M1
    societe: str
      VALUE: Northgate Hardware
    type: str
      VALUE: weekly operations meeting
    date: str
      VALUE: 2026-08-03
    participants: list
      LIST (4 items)
        VALUE: Rafael
    repliques: list
      LIST (18 items)
        DICT (2 keys)
        qui: str
        texte: str
    verite_terrain: dict
      DICT (2 keys)
      actions: list
        LIST (5 items)
      pieges: list
        LIST (3 items)
compte: dict
  DICT (5 keys)
  reunions: int
    VALUE: 8
  actions_reelles: int
    VALUE: 40
  actions_attribuees: int
    VALUE: 24
  actions_non_attribuees: int
    VALUE: 16
  pieges: int
    VALUE: 24


In [15]:
def count_structures(obj):

    dictionaries = 0
    lists = 0

    if isinstance(obj, dict):

        dictionaries += 1

        for value in obj.values():

            d, l = count_structures(
                value
            )

            dictionaries += d
            lists += l

    elif isinstance(obj, list):

        lists += 1

        for value in obj:

            d, l = count_structures(
                value
            )

            dictionaries += d
            lists += l

    return dictionaries, lists


dictionary_count, list_count = (
    count_structures(corpus)
)

print(
    "Dictionaries:",
    dictionary_count
)

print(
    "Lists:",
    list_count
)

Dictionaries: 236
Lists: 33


In [16]:
def find_lists(
    obj,
    path="root"
):

    results = []

    if isinstance(obj, dict):

        for key, value in obj.items():

            current_path = (
                f"{path}.{key}"
            )

            if isinstance(
                value,
                list
            ):

                results.append({
                    "path": current_path,
                    "length": len(value),
                    "type": "list"
                })

            results.extend(
                find_lists(
                    value,
                    current_path
                )
            )

    elif isinstance(obj, list):

        for index, value in enumerate(obj):

            current_path = (
                f"{path}[{index}]"
            )

            results.extend(
                find_lists(
                    value,
                    current_path
                )
            )

    return results


list_locations = find_lists(
    corpus
)

pd.DataFrame(
    list_locations
)

,path,length,type
0,root.reunions,8,list
1,root.reunions[0].participants,4,list
2,root.reunions[0].repliques,18,list
3,root.reunions[0].verite_terrain.actions,5,list
4,root.reunions[0].verite_terrain.pieges,3,list
5,root.reunions[1].participants,4,list
6,root.reunions[1].repliques,20,list
7,root.reunions[1].verite_terrain.actions,5,list
8,root.reunions[1].verite_terrain.pieges,3,list
9,root.reunions[2].participants,4,list


In [17]:
def find_fields(
    obj,
    keywords,
    path="root"
):

    results = []

    if isinstance(obj, dict):

        for key, value in obj.items():

            current_path = (
                f"{path}.{key}"
            )

            key_text = str(
                key
            ).lower()

            if any(
                keyword in key_text
                for keyword in keywords
            ):

                results.append({
                    "path": current_path,
                    "key": key,
                    "type": type(value).__name__,
                    "preview": str(value)[:300]
                })

            results.extend(
                find_fields(
                    value,
                    keywords,
                    current_path
                )
            )

    elif isinstance(obj, list):

        for index, value in enumerate(obj):

            current_path = (
                f"{path}[{index}]"
            )

            results.extend(
                find_fields(
                    value,
                    keywords,
                    current_path
                )
            )

    return results


transcript_fields = find_fields(
    corpus,
    [
        "transcript",
        "text",
        "meeting",
        "dialogue",
        "conversation",
        "speaker"
    ]
)

pd.DataFrame(
    transcript_fields
)

,path,key,type,preview
0,root.reunions[0].repliques[0].texte,texte,str,"Right, let us get started, I have a hard stop ..."
1,root.reunions[0].repliques[1].texte,texte,str,"Two people are off next week, so we will be sh..."
2,root.reunions[0].repliques[2].texte,texte,str,I will update the pricing on the kiosk by Wedn...
3,root.reunions[0].repliques[3].texte,texte,str,I will take down the summer banner.
4,root.reunions[0].repliques[4].texte,texte,str,"Nadia, can you call the sign shop about the aw..."
...,...,...,...,...
149,root.reunions[7].repliques[16].texte,texte,str,"Two people are off next week, so we will be sh..."
150,root.reunions[7].repliques[17].texte,texte,str,"I already returned the floor mats last week, t..."
151,root.reunions[7].repliques[18].texte,texte,str,Someone has to see to this: the yard floodligh...
152,root.reunions[7].repliques[19].texte,texte,str,Anything else before we close?


In [18]:
action_fields = find_fields(
    corpus,
    [
        "action",
        "task",
        "todo",
        "owner",
        "deadline",
        "due",
        "assignee",
        "ground",
        "expected"
    ]
)

pd.DataFrame(
    action_fields
)

,path,key,type,preview
0,root.reunions[0].verite_terrain.actions,actions,list,"[{'code': 'A1', 'ancre': 'kiosk', 'famille': '..."
1,root.reunions[1].verite_terrain.actions,actions,list,"[{'code': 'A1', 'ancre': 'hydrant', 'famille':..."
2,root.reunions[2].verite_terrain.actions,actions,list,"[{'code': 'A1', 'ancre': 'shredder', 'famille'..."
3,root.reunions[3].verite_terrain.actions,actions,list,"[{'code': 'A1', 'ancre': 'gutters', 'famille':..."
4,root.reunions[4].verite_terrain.actions,actions,list,"[{'code': 'A1', 'ancre': 'dumpster', 'famille'..."
5,root.reunions[5].verite_terrain.actions,actions,list,"[{'code': 'A1', 'ancre': 'thermostat', 'famill..."
6,root.reunions[6].verite_terrain.actions,actions,list,"[{'code': 'A1', 'ancre': 'lockers', 'famille':..."
7,root.reunions[7].verite_terrain.actions,actions,list,"[{'code': 'A1', 'ancre': 'badges', 'famille': ..."
8,root.compte.actions_reelles,actions_reelles,int,40
9,root.compte.actions_attribuees,actions_attribuees,int,24


In [19]:
summary = {
    "root_type": type(corpus).__name__,
    "root_keys": len(corpus),
    "dictionary_count": dictionary_count,
    "list_count": list_count,
    "transcript_fields_found": len(
        transcript_fields
    ),
    "action_fields_found": len(
        action_fields
    )
}

summary_df = pd.DataFrame(
    [
        summary
    ]
)

summary_df

,root_type,root_keys,dictionary_count,list_count,transcript_fields_found,action_fields_found
0,dict,3,236,33,154,11


In [20]:
print(
    json.dumps(
        corpus,
        indent=2,
        ensure_ascii=False
    )[:10000]
)

{
  "graine": 20260818,
  "reunions": [
    {
      "id": "M1",
      "societe": "Northgate Hardware",
      "type": "weekly operations meeting",
      "date": "2026-08-03",
      "participants": [
        "Rafael",
        "Nadia",
        "Colin",
        "Padma"
      ],
      "repliques": [
        {
          "qui": "Rafael",
          "texte": "Right, let us get started, I have a hard stop at eleven."
        },
        {
          "qui": "Nadia",
          "texte": "Two people are off next week, so we will be short handed."
        },
        {
          "qui": "Rafael",
          "texte": "I will update the pricing on the kiosk by Wednesday."
        },
        {
          "qui": "Colin",
          "texte": "I will take down the summer banner."
        },
        {
          "qui": "Colin",
          "texte": "Nadia, can you call the sign shop about the awning?"
        },
        {
          "qui": "Nadia",
          "texte": "Sure, that is mine."
        },
        {
        